# Knowledge Integration in Task-Tuned Financial LLMs

**Setup notebook for Google Colab.** Clones the repo, installs dependencies,
mounts Google Drive for persistent data/checkpoints, and verifies GPU access.

Recommended runtime: **A100 GPU** (Colab Pro) or **T4** (free tier).

## 1. Mount Google Drive

We use Drive to persist the FNSPID dataset (~22GB), checkpoints, and outputs
across Colab sessions.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Create persistent directories on Drive
!mkdir -p /content/drive/MyDrive/sot/{data,checkpoints,outputs}

## 2. Clone repo and install

In [ ]:
!git clone --recurse-submodules https://github.com/jeremiahmao/still-on-task-llm.git
%cd still-on-task-llm
!pip install -e . -q

In [ ]:
# Symlink persistent storage into the repo
from pathlib import Path

links = {
    "data": "/content/drive/MyDrive/sot/data",
    "checkpoints": "/content/drive/MyDrive/sot/checkpoints",
    "outputs": "/content/drive/MyDrive/sot/outputs",
}

for local, remote in links.items():
    local_path = Path(local)
    if local_path.exists() and not local_path.is_symlink():
        # Move any existing local data to Drive first
        if any(local_path.iterdir()):
            !cp -rn {local}/* {remote}/ 2>/dev/null || true
        !rm -rf {local}
    if not local_path.exists():
        local_path.symlink_to(remote)
        print(f"{local} -> {remote}")

## 3. Verify environment

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Verify all update methods import
print("\nAll 4 update methods imported OK")

## 4. Phase 0 — Data pipeline

Run these once. Results persist on Google Drive.

In [ ]:
# Step 1: Download datasets (FNSPID ~22GB, FinQA ~50MB)
# If you already ran this locally and uploaded to Drive, skip this cell.
!python scripts/01_download_data.py

In [ ]:
# Step 2: Build retrieval corpus from pre-cutoff FNSPID articles
!python scripts/02_build_corpus.py

In [ ]:
# Step 3: Build FAISS index for retrieval evaluation
!python scripts/03_build_faiss_index.py

## 5. Task-tune base model

Fine-tune Qwen2.5-3B on query decomposition via LoRA SFT.
This produces the baseline checkpoint that all update methods start from.

**Prerequisite:** QD training data must exist at `data/qd/train.json`.

In [ ]:
# TODO: Run task-tuning script (or upload pre-trained checkpoint to Drive)
# !python scripts/04_task_tune_qd.py  # if this script exists
#
# Otherwise, verify checkpoint exists:
from pathlib import Path

ckpt = Path("checkpoints/qd_sft/final")
if ckpt.exists():
    print(f"Checkpoint found at {ckpt}")
else:
    print(f"WARNING: No checkpoint at {ckpt}. Task-tuning must run first.")

## 6. Phase 1 — Core comparison at 1K edits

Runs 4 update methods + no-update baseline on the query decomposition task.
This is the minimum viable result for the paper.

In [ ]:
!python scripts/11_run_phase1.py

## 7. Phase 2 — Scaling to 3K edits

In [ ]:
!python scripts/12_run_phase2.py

## 8. Phase 3 — FinQA forgetting control (time permitting)

In [ ]:
!python scripts/13_run_phase3.py

## 9. Generate results tables

In [ ]:
!python scripts/14_generate_tables.py

## Tips

- **Resuming after disconnect:** Re-run cells 1-3 (mount, clone, symlink). Your data/checkpoints/outputs are on Drive.
- **Running Phase 0 locally:** The data pipeline is CPU-only. Run it on your Mac, then upload `data/` to `sot/data/` on Drive.
- **WandB logging:** Run `!wandb login` before experiments to enable logging. Or set `wandb.enabled: false` in `configs/base.yaml`.
- **Individual methods:** To run a single method instead of the full phase:
  ```
  !python scripts/09_run_update.py --method copr --scale 1000 --task qd --config configs/update/copr.yaml
  !python scripts/10_evaluate.py --model_path outputs/copr_qd_scale1000 --task qd --metrics preservation,absorption,locality
  ```